In [ ]:
# 04_groundwater — CELL 0
import pandas as pd, numpy as np, re, statsmodels.api as sm
from google.colab import drive; drive.mount('/content/drive')
BASE = '/content/drive/My Drive/Data/Elevar'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
DROP = ['NCT Delhi','Lakshadweep Islands','Andaman & Nicobar','Chandigarh',
        'Daman & Diu','Dadra & Nagar Haveli','Puducherry','Goa','Himachal Pradesh',
        'Sikkim','Tripura','Manipur','Mizoram','Nagaland','Meghalaya','Arunachal Pradesh','Jammu & Kashmir']

# SHOT 1A — verify the CGWB file and match it to ICRISAT parents
import pandas as pd, numpy as np, re
BASE = '/content/drive/My Drive/Data/Elevar'

GW = pd.read_csv(f'{BASE}/cgwb_district_2011.csv')
print('raw:', GW.shape); print(GW.columns.tolist())
GW['soe_pct'] = pd.to_numeric(GW.soe_pct, errors='coerce')
GW = GW.dropna(subset=['soe_pct'])
GW['category'] = pd.cut(GW.soe_pct, [-1,70,90,100,1e6],
                        labels=['Safe','Semi-Critical','Critical','Over-Exploited'])
print('\nsoe_pct:', GW.soe_pct.describe().round(1).to_dict())
print('\ncategory counts:'); print(GW.category.value_counts().to_string())
print('national SoE (unweighted mean):', round(GW.soe_pct.mean(),1), '| 2011 published national ~62%')

GW = GW[~GW.state.isin(DROP)].copy()
GW['soe_pct'] = pd.to_numeric(GW.soe_pct, errors='coerce')
GW = GW[GW.soe_pct.between(0, 400)]          # Chennai/Kala Amb style outliers, none in sample
GW['category'] = pd.cut(GW.soe_pct, [-1,70,90,100,1e6],
                        labels=['Safe','Semi-Critical','Critical','Over-Exploited'])
print('usable district rows:', len(GW), '| states:', GW.state.nunique())

def norm(s):
    s = str(s).lower().strip()
    s = re.sub(r'\b(district|dist|urban|rural)\b','',s)
    return re.sub(r'[^a-z]','',s)

GW['key'] = GW.district.map(norm)
GW['skey'] = GW.state.map(norm)

H1 = pd.read_parquet(f'{BASE}/h1_era_exposure.parquet')
H1['key']  = H1['Dist Name'].map(norm)
H1['skey'] = H1['State Name'].map(norm)

# match within state first, then nationally for the leftovers
m1 = H1.merge(GW[['key','skey','soe_pct','category']], on=['key','skey'], how='left')
miss = m1.soe_pct.isna()
m2 = m1[miss].drop(columns=['soe_pct','category']).merge(
        GW.drop_duplicates('key')[['key','soe_pct','category']], on='key', how='left')
M = pd.concat([m1[~miss], m2], ignore_index=True)
print(f'\nmatched {M.soe_pct.notna().sum()} of {len(H1)} H1 districts')
print('unmatched:', M[M.soe_pct.isna()]['Dist Name'].tolist()[:25])

M.to_parquet(f'{BASE}/h1_with_gw.parquet', index=False)
print('\nsaved h1_with_gw.parquet')
print(M.dropna(subset=["soe_pct"]).nlargest(10,'soe_pct')[['Dist Name','State Name','soe_pct','category','d_exposure']].round(2).to_string(index=False))

raw: (638, 9)
['state', 'district', 'year', 'annual_extractable_gw', 'annual_gw_extraction', 'soe_pct', 'category', 'units', 'flag']

soe_pct: {'count': 638.0, 'mean': 63.7, 'std': 54.8, 'min': 0.0, '25%': 27.2, '50%': 54.0, '75%': 83.8, 'max': 565.0}

category counts:
category
Safe              423
Over-Exploited    113
Semi-Critical      77
Critical           25
national SoE (unweighted mean): 63.7 | 2011 published national ~62%
usable district rows: 514 | states: 18

matched 196 of 248 H1 districts
unmatched: ['Bastar', 'Seoni / Shivani', 'Khargone / West Nimar', 'Khandwa / East Nimar', 'S.P.S. Nellore', 'Ananthapur', 'Kadapa YSR', 'Hyderabad', 'Mahabubnagar', 'Shimoge', 'Chickmagalur', 'Bijapur / Vijayapura', 'Gulbarga / Kalaburagi', 'Dakshina Kannada', 'Uttara Kannada', 'Kodagu / Coorg', 'South Arcot / Cuddalore', 'North Arcot / Vellore', 'Tiruchirapalli / Trichy', 'Ramananthapuram', 'Thirunelveli', 'The Nilgiris', 'Nasik', 'Amarawati', 'Vadodara / Baroda']

saved h1_with_gw.parqu

In [ ]:
import pandas as pd, numpy as np, re, statsmodels.api as sm
from rapidfuzz import process, fuzz
BASE = '/content/drive/My Drive/Data/Elevar'

M  = pd.read_parquet(f'{BASE}/h1_with_gw.parquet')
GW = pd.read_csv(f'{BASE}/cgwb_district_2011.csv')
DROP = ['NCT Delhi','Lakshadweep Islands','Andaman & Nicobar','Chandigarh','Daman & Diu',
        'Dadra & Nagar Haveli','Puducherry','Goa','Himachal Pradesh','Sikkim','Tripura',
        'Manipur','Mizoram','Nagaland','Meghalaya','Arunachal Pradesh','Jammu & Kashmir']
GW = GW[~GW.state.isin(DROP)].copy()
GW['soe_pct'] = pd.to_numeric(GW.soe_pct, errors='coerce')
GW = GW[GW.soe_pct.between(0,400)]

def norm(s):
    s = str(s).lower()
    s = re.split(r'[/(]', s)[0]                      # "Seoni / Shivani" -> "seoni"
    s = re.sub(r'\b(district|dist|the|urban|rural)\b','',s)
    for a,b in [('sh','s'),('ph','f'),('th','t'),('kh','k'),('gh','g'),('bh','b'),('dh','d'),
                ('oo','u'),('ee','i'),('aa','a'),('w','v'),('z','j'),('y','i')]:
        s = s.replace(a,b)
    return re.sub(r'[^a-z]','',s)

GW['key'] = GW.district.map(norm)
un = M[M.soe_pct.isna()].copy()
un['key'] = un['Dist Name'].map(norm)

# also try the ALTERNATE name after the slash
def alt(s):
    parts = re.split(r'[/(]', str(s))
    return norm(parts[1]) if len(parts) > 1 else None
un['key2'] = un['Dist Name'].map(alt)

pool = GW.drop_duplicates('key')
fixes = []
for _, r in un.iterrows():
    cand = pool[pool.state.map(norm) == norm(r['State Name'])]
    if not len(cand): cand = pool
    best = None
    for k in [r['key'], r['key2']]:
        if not k: continue
        m = process.extractOne(k, cand['key'].tolist(), scorer=fuzz.WRatio)
        if m and (best is None or m[1] > best[1]): best = m
    if best and best[1] >= 80:
        row = cand.iloc[best[2]]
        fixes.append({'Dist Code': r['Dist Code'], 'soe_pct': row.soe_pct,
                      'matched_to': row.district, 'score': round(best[1],1)})
FX = pd.DataFrame(fixes)
print(f'recovered {len(FX)} of {len(un)}')
print(FX.merge(un[['Dist Code','Dist Name']], on='Dist Code')[['Dist Name','matched_to','score','soe_pct']].to_string(index=False))

M = M.set_index('Dist Code')
M.loc[FX['Dist Code'], 'soe_pct'] = FX.set_index('Dist Code').soe_pct
M['category'] = pd.cut(M.soe_pct, [-1,70,90,100,1e6],
                       labels=['Safe','Semi-Critical','Critical','Over-Exploited'])
M['improved'] = M.d_exposure < 0
print(f'\nTOTAL MATCHED: {M.soe_pct.notna().sum()} of {len(M)}')

# ---------------- THE RESULT ----------------
S = M.dropna(subset=['soe_pct'])
print('\n=== GROUNDWATER STRESS BY RESILIENCE OUTCOME ===')
print(pd.crosstab(S.improved, S.category, normalize='index').round(3).to_string())
print(f'\nmean SoE  improvers {S[S.improved].soe_pct.mean():.1f}%  |  non-improvers {S[~S.improved].soe_pct.mean():.1f}%')
print(f'corr(d_exposure, soe_pct) = {S.d_exposure.corr(S.soe_pct):+.3f}   (negative = bigger gains where extraction higher)')
print(f'rank corr = {S.d_exposure.corr(S.soe_pct, method="spearman"):+.3f}')

imp = S[S.improved]
print('\n=== HEADLINE ===')
print(f'  of {len(imp)} districts that became more drought-resilient:')
print(f'    {(imp.soe_pct>100).sum():3d} ({(imp.soe_pct>100).mean():.0%}) over-exploited (>100%)')
print(f'    {(imp.soe_pct>70).sum():3d} ({(imp.soe_pct>70).mean():.0%}) stressed (>70%)')
non = S[~S.improved]
print(f'  vs non-improvers: {(non.soe_pct>100).mean():.0%} over-exploited, {(non.soe_pct>70).mean():.0%} stressed')

# top quartile of improvement
q = S[S.d_exposure <= S.d_exposure.quantile(0.25)]
print(f'\n  TOP QUARTILE of improvement (n={len(q)}): mean SoE {q.soe_pct.mean():.1f}%, '
      f'{(q.soe_pct>100).mean():.0%} over-exploited')

# ---------------- regression ----------------
R = pd.read_parquet(f'{BASE}/h1_regression_frame.parquet')
print(R.columns) # Added line to diagnose the error
R = R.set_index('Dist Code') # Corrected 'dist_code' to 'Dist Code'
Q = S.join(R[['d_irr_share','d_tubewell_share','d_canal_share']], how='left').dropna(
        subset=['d_exposure','soe_pct','d_tubewell_share','d_irr_share','exp_e1'])
X = Q[['soe_pct','d_tubewell_share','d_canal_share','d_irr_share','exp_e1']]
X = (X-X.mean())/X.std()
m = sm.OLS(Q.d_exposure.values, sm.add_constant(X)).fit(cov_type='HC3')
print(f'\nΔexposure ~ groundwater stress + irrigation composition (standardised, N {len(Q)}, R2 {m.rsquared:.3f})')
o = pd.DataFrame({'coef':m.params,'t':m.tvalues,'p':m.pvalues}).drop('const')
print(o.reindex(o.t.abs().sort_values(ascending=False).index).round(3).to_string())

M.reset_index().to_parquet(f'{BASE}/shot1_groundwater.parquet', index=False)
print('\nsaved shot1_groundwater.parquet')

recovered 42 of 52
                    Dist Name      matched_to  score  soe_pct
                       Bastar          Baster   83.3     13.0
              Seoni / Shivani           Seoni  100.0     29.0
        Khargone / West Nimar        Khargone  100.0     75.0
         Khandwa / East Nimar         Khandwa  100.0     62.0
               S.P.S. Nellore         Nellore   82.4     59.0
                   Ananthapur       Anantapur  100.0     62.0
                   Kadapa YSR          Kadapa   90.0     27.0
                 Mahabubnagar   Mehaboobnagar   91.7     59.0
                      Shimoge         Shimoga   83.3     27.0
                 Chickmagalur     Chikmagalur   95.7     43.0
        Gulbarga / Kalaburagi        Gulbarga  100.0     32.0
             Dakshina Kannada Dakshin Kannada   96.3     69.0
               Uttara Kannada   Uttar kannada   96.0     37.0
               Kodagu / Coorg          Kodagu  100.0     21.0
      South Arcot / Cuddalore       Cuddalore  100.

In [ ]:
# 04 CELL 3 — drop bad matches, re-test, add the conditional version
import pandas as pd, numpy as np, statsmodels.api as sm
BASE = '/content/drive/My Drive/Data/Elevar'
M = pd.read_parquet(f'{BASE}/shot1_groundwater.parquet').set_index('Dist Code')

BAD = ['Kangra','Mandi','North Cachar Hil / Dima hasao']
M.loc[M['Dist Name'].isin(BAD), 'soe_pct'] = np.nan
M['category'] = pd.cut(M.soe_pct,[-1,70,90,100,1e6],
                       labels=['Safe','Semi-Critical','Critical','Over-Exploited'])
S = M.dropna(subset=['soe_pct']); S = S.assign(improved=S.d_exposure<0)
print('clean N =', len(S))
print(pd.crosstab(S.improved, S.category, normalize='index').round(3).to_string())
imp, non = S[S.improved], S[~S.improved]
print(f'\nover-exploited: improvers {(imp.soe_pct>100).mean():.1%} vs non {(non.soe_pct>100).mean():.1%}')
print(f'stressed >70%:  improvers {(imp.soe_pct>70).mean():.1%} vs non {(non.soe_pct>70).mean():.1%}')

# is the 20% vs 11% difference statistically real?
from scipy.stats import chi2_contingency, fisher_exact
ct = pd.crosstab(S.improved, S.soe_pct>100)
chi2,p,_,_ = chi2_contingency(ct); odds,pf = fisher_exact(ct)
print(f'\nover-exploited x improved: chi2 p={p:.4f} | Fisher OR={odds:.2f}, p={pf:.4f}')

# the conditional claim, done properly: logit of over-exploited on improvement
S2 = S.dropna(subset=['exp_e1'])
X = sm.add_constant(pd.DataFrame({'improved':S2.improved.astype(float),
                                  'exp_e1':(S2.exp_e1-S2.exp_e1.mean())/S2.exp_e1.std()}))
lg = sm.Logit((S2.soe_pct>100).astype(float), X).fit(disp=0)
print(f'\nP(over-exploited) ~ improved + initial exposure, N {len(S2)}')
print(pd.DataFrame({'coef':lg.params,'t':lg.tvalues,'p':lg.pvalues}).round(3).to_string())
print('  marginal effect of "improved":', round(lg.get_margeff().margeff[0],3))

M.reset_index().to_parquet(f'{BASE}/shot1_groundwater.parquet', index=False)

clean N = 235
category   Safe  Semi-Critical  Critical  Over-Exploited
improved                                                
False     0.670          0.125     0.091           0.114
True      0.612          0.184     0.007           0.197

over-exploited: improvers 19.7% vs non 11.4%
stressed >70%:  improvers 38.8% vs non 33.0%

over-exploited x improved: chi2 p=0.1370 | Fisher OR=1.92, p=0.1058

P(over-exploited) ~ improved + initial exposure, N 235
           coef      t      p
const    -1.979 -5.838  0.000
improved  0.500  1.225  0.221
exp_e1    0.254  1.565  0.118
  marginal effect of "improved": 0.068
